In [8]:
import json
import re

import pandas as pd
from datasets import load_dataset

In [33]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [9]:
# Set this to the dataset you want to inspect
REPO_ID = "eturok-weizmann/laser-vibrations"

In [10]:
# Load metadata only
ds = load_dataset(
    REPO_ID,
    split="train",
    verification_mode="no_checks",
)

In [11]:
ds

Dataset({
    features: ['sample_id', 'segmented_overhead_file_name', 'speckle_vibrations_file_name', 'speckle_shifts_ifft_audio_file_name', 'audio_file_name', 'experiment_id', 'speakers', 'x_position', 'y_position', 'x_com', 'y_com', 'object', 'n_objects', 'box_material', 'mask_file_name', 'experiment_dir', 'manifest'],
    num_rows: 461
})

In [12]:
# Convert to pandas for easy inspection
df = ds.to_pandas()

In [13]:
# 1. Which ID column exists?
id_candidates = ["sample_id", "sample_idx"]
id_presence = pd.DataFrame(
    {
        "column": id_candidates,
        "exists": [col in df.columns for col in id_candidates],
    }
)

In [14]:
id_presence

,column,exists
0,sample_id,True
1,sample_idx,False


In [15]:
# 2. Unique values for speakers
if "speakers" in df.columns:
    speakers_df = (
        df["speakers"]
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis("speakers")
        .reset_index(name="count")
        .sort_values(["count", "speakers"], ascending=[False, True])
        .reset_index(drop=True)
    )
else:
    speakers_df = pd.DataFrame(columns=["speakers", "count"])

In [16]:
speakers_df

,speakers,count
0,0001,116
1,0010,115
2,0100,115
3,1000,115


In [17]:
# 3. Counts per (x_position, y_position, speakers)
group_cols = ["x_position", "y_position", "speakers"]
if all(col in df.columns for col in group_cols):
    position_speaker_counts_df = (
        df.groupby(group_cols, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["x_position", "y_position", "speakers"])
        .reset_index(drop=True)
    )
else:
    position_speaker_counts_df = pd.DataFrame(columns=group_cols + ["count"])

In [18]:
position_speaker_counts_df

,x_position,y_position,speakers,count
0,0.0,1.0,0001,1
1,0.0,1.0,0010,1
2,0.0,1.0,0100,1
3,0.0,1.0,1000,1
4,0.0,2.0,0001,1
...,...,...,...,...
415,10.0,11.0,1000,1
416,NaN,NaN,0001,8
417,NaN,NaN,0010,8
418,NaN,NaN,0100,8


In [19]:
# 4. Counts per n_objects
if "n_objects" in df.columns:
    n_objects_df = (
        df["n_objects"]
        .value_counts(dropna=False)
        .rename_axis("n_objects")
        .reset_index(name="count")
        .sort_values("n_objects")
        .reset_index(drop=True)
    )
else:
    n_objects_df = pd.DataFrame(columns=["n_objects", "count"])

In [20]:
n_objects_df

,n_objects,count
0,0,4
1,1,457


In [21]:
# 5. Whether there is any explicit flag/field for off-grid samples
offgrid_name_patterns = [
    r"off.?grid",
    r"between",
    r"interpol",
    r"subgrid",
    r"fractional",
    r"offset",
]
offgrid_cols = [
    col for col in df.columns
    if any(re.search(pat, col, flags=re.IGNORECASE) for pat in offgrid_name_patterns)
]
offgrid_columns_df = pd.DataFrame({"offgrid_like_column": offgrid_cols})

In [22]:
offgrid_columns_df

,offgrid_like_column


In [23]:
# Also inspect manifest keys if manifest exists, without fully normalizing everything
manifest_key_counts = {}
if "manifest" in df.columns:
    for raw in df["manifest"].dropna():
        try:
            m = json.loads(raw) if isinstance(raw, str) else raw
            if isinstance(m, dict):
                for k in m.keys():
                    manifest_key_counts[k] = manifest_key_counts.get(k, 0) + 1
        except Exception:
            pass

manifest_keys_df = (
    pd.DataFrame(
        sorted(manifest_key_counts.items(), key=lambda kv: (-kv[1], kv[0])),
        columns=["manifest_key", "rows_present"],
    )
    if manifest_key_counts
    else pd.DataFrame(columns=["manifest_key", "rows_present"])
)

manifest_offgrid_keys_df = manifest_keys_df[
    manifest_keys_df["manifest_key"].str.contains(
        "|".join(offgrid_name_patterns),
        case=False,
        na=False,
        regex=True,
    )
].reset_index(drop=True)

In [24]:
manifest_offgrid_keys_df

,manifest_key,rows_present


In [25]:
# Experiment ID format / unique values
experiment_id_col = None
for candidate in ["experiment_id", "source_experiment_id"]:
    if candidate in df.columns:
        experiment_id_col = candidate
        break

if experiment_id_col is not None:
    experiment_ids = df[experiment_id_col].dropna().astype(str)

    experiment_id_examples_df = pd.DataFrame(
        {
            "example_experiment_id": sorted(experiment_ids.unique())[:50]
        }
    )

    experiment_id_patterns_df = (
        experiment_ids
        .str.extract(r"^(.*?)(?:__|--|$)", expand=False)
        .fillna("<missing>")
        .value_counts()
        .rename_axis("experiment_id_prefix")
        .reset_index(name="count")
        .sort_values(["count", "experiment_id_prefix"], ascending=[False, True])
        .reset_index(drop=True)
    )
else:
    experiment_id_examples_df = pd.DataFrame(columns=["example_experiment_id"])
    experiment_id_patterns_df = pd.DataFrame(columns=["experiment_id_prefix", "count"])

In [35]:
experiment_id_patterns_df

,experiment_id_prefix,count
0,cube-07x04y_0001,2
1,cube-07x04y_0010,2
2,cube-07x04y_0100,2
3,cube-07x04y_1000,2
4,cube-08x01y_0001,2
5,cube-08x01y_0010,2
6,cube-08x01y_0100,2
7,cube-08x01y_1000,2
8,cube-09x00y_0001,2
9,cube-09x00y_0010,2


In [ ]:
# Compact summary table
summary_df = pd.DataFrame(
    [
        {"check": "n_rows", "value": len(df)},
        {"check": "n_columns", "value": len(df.columns)},
        {"check": "has_sample_id", "value": "sample_id" in df.columns},
        {"check": "has_sample_idx", "value": "sample_idx" in df.columns},
        {"check": "n_unique_speakers", "value": df["speakers"].nunique(dropna=False) if "speakers" in df.columns else None},
        {"check": "n_unique_positions", "value": df[["x_position", "y_position"]].drop_duplicates().shape[0] if all(c in df.columns for c in ["x_position", "y_position"]) else None},
        {"check": "n_unique_position_speaker", "value": df[["x_position", "y_position", "speakers"]].drop_duplicates().shape[0] if all(c in df.columns for c in ["x_position", "y_position", "speakers"]) else None},
        {"check": "n_unique_n_objects", "value": df["n_objects"].nunique(dropna=False) if "n_objects" in df.columns else None},
        {"check": "n_offgrid_like_columns", "value": len(offgrid_cols)},
        {"check": "experiment_id_column", "value": experiment_id_col},
    ]
)

display(summary_df)
display(id_presence)
display(speakers_df)
display(n_objects_df)
display(position_speaker_counts_df)
display(offgrid_columns_df)
display(manifest_offgrid_keys_df)
display(manifest_keys_df.head(50))
display(experiment_id_patterns_df.head(50))
display(experiment_id_examples_df.head(50))
